# SDH exp_003 — 전처리 10종 벤치마크

공용 `run_preprocessing_benchmark()`를 사용하여 모델과 5-Fold 조건은 고정하고 전처리만 비교합니다.

- 1차 실행: seed 42
- 평가 기준: OOF Macro F1
- 유망 후보만 seed 42/52/62로 반복 검증
- 빈도 필터와 hotspot은 각 fold의 train 데이터에서만 학습

In [2]:
from pathlib import Path
import sys

search_paths = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(
    (
        path
        for path in search_paths
        if (path / "common").is_dir()
        and (path / "experiments").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from experiments.SDH.exp_003_preprocessing.preprocessing import (
    make_preprocessing_candidates,
)
from experiments.SDH.exp_003_preprocessing.run_benchmark import run

print(f"프로젝트 루트: {PROJECT_ROOT}")

프로젝트 루트: /mnt/c/dev/final_hackaton


## 전처리 후보 확인

| Case | 전처리 |
| --- | --- |
| 01 | 결측→WT, WT/변이 이진화 |
| 02 | 01 + 상수 유전자 제거 |
| 03 | 02 + `log1p(변이 유전자 수)` |
| 04 | 02 + `log1p(실제 변이 토큰 수)` |
| 05 | 02 + 두 burden |
| 06 | 05 + 변이 유형별 개수 |
| 07 | 05 + 최소 변이 빈도 3 |
| 08 | 05 + 최소 변이 빈도 5 |
| 09 | 05 + 최소 변이 빈도 10 |
| 10 | 05 + 상위 50개 mutation-token hotspot |

In [3]:
candidates = make_preprocessing_candidates()
list(candidates)

['case_01_wt_binary',
 'case_02_remove_constant',
 'case_03_gene_burden',
 'case_04_token_burden',
 'case_05_both_burdens',
 'case_06_mutation_types',
 'case_07_min_count_3',
 'case_08_min_count_5',
 'case_09_min_count_10',
 'case_10_hotspot_top50']

## 빠른 핵심 비교

먼저 baseline과 burden 관련 네 후보를 비교합니다. 각 후보마다 5-Fold 학습이 실행됩니다.

In [4]:
core_cases = [
    "case_01_wt_binary",
    "case_03_gene_burden",
    "case_04_token_burden",
    "case_05_both_burdens",
]

core_leaderboard = run(selected_cases=core_cases)
core_leaderboard[
    [
        "preprocessing",
        "oof_f1_macro_mean",
        "oof_accuracy_mean",
        "fold_f1_macro_std",
        "elapsed_seconds",
    ]
]


===== case_01_wt_binary =====
[logistic] seed=42 fold=1/5 f1_macro=0.34554 features=4,384 time=15.3s
[logistic] seed=42 fold=2/5 f1_macro=0.32894 features=4,384 time=12.7s
[logistic] seed=42 fold=3/5 f1_macro=0.35267 features=4,384 time=12.1s
[logistic] seed=42 fold=4/5 f1_macro=0.34190 features=4,384 time=13.3s
[logistic] seed=42 fold=5/5 f1_macro=0.34704 features=4,384 time=12.5s
[logistic] seed=42 OOF Macro F1=0.34452, Accuracy=0.34285
완료: OOF Macro F1 0.34452
fold Macro F1 0.34322 ± 0.00887

===== case_03_gene_burden =====
[logistic] seed=42 fold=1/5 f1_macro=0.37033 features=4,231 time=23.5s
[logistic] seed=42 fold=2/5 f1_macro=0.35227 features=4,230 time=24.5s
[logistic] seed=42 fold=3/5 f1_macro=0.36464 features=4,231 time=23.3s
[logistic] seed=42 fold=4/5 f1_macro=0.35831 features=4,229 time=23.8s
[logistic] seed=42 fold=5/5 f1_macro=0.36121 features=4,227 time=22.8s
[logistic] seed=42 OOF Macro F1=0.36238, Accuracy=0.35962
완료: OOF Macro F1 0.36238
fold Macro F1 0.36135 ± 0.00

,preprocessing,oof_f1_macro_mean,oof_accuracy_mean,fold_f1_macro_std,elapsed_seconds
0,case_05_both_burdens,0.362392,0.360748,0.007648,166.223182
1,case_03_gene_burden,0.362379,0.359619,0.006767,117.864923
2,case_04_token_burden,0.360935,0.358652,0.006160,140.068935
3,case_01_wt_binary,0.344525,0.342848,0.008874,66.213901


## 10개 전체 비교

아래 셀은 10개 후보를 모두 실행하므로 시간이 오래 걸릴 수 있습니다.

In [5]:
leaderboard = run()
leaderboard[
    [
        "preprocessing",
        "oof_f1_macro_mean",
        "oof_accuracy_mean",
        "fold_f1_macro_std",
        "elapsed_seconds",
    ]
]


===== case_01_wt_binary =====
[logistic] seed=42 fold=1/5 f1_macro=0.34554 features=4,384 time=12.9s
[logistic] seed=42 fold=2/5 f1_macro=0.32894 features=4,384 time=12.5s
[logistic] seed=42 fold=3/5 f1_macro=0.35267 features=4,384 time=12.2s
[logistic] seed=42 fold=4/5 f1_macro=0.34190 features=4,384 time=13.3s
[logistic] seed=42 fold=5/5 f1_macro=0.34704 features=4,384 time=12.9s
[logistic] seed=42 OOF Macro F1=0.34452, Accuracy=0.34285
완료: OOF Macro F1 0.34452
fold Macro F1 0.34322 ± 0.00887

===== case_02_remove_constant =====
[logistic] seed=42 fold=1/5 f1_macro=0.34554 features=4,230 time=12.2s
[logistic] seed=42 fold=2/5 f1_macro=0.32894 features=4,229 time=12.3s
[logistic] seed=42 fold=3/5 f1_macro=0.35235 features=4,230 time=13.0s
[logistic] seed=42 fold=4/5 f1_macro=0.34147 features=4,228 time=12.3s
[logistic] seed=42 fold=5/5 f1_macro=0.34704 features=4,226 time=12.7s
[logistic] seed=42 OOF Macro F1=0.34439, Accuracy=0.34269
완료: OOF Macro F1 0.34439
fold Macro F1 0.34307 ± 

,preprocessing,oof_f1_macro_mean,oof_accuracy_mean,fold_f1_macro_std,elapsed_seconds
0,case_06_mutation_types,0.378027,0.374133,0.012054,190.849165
1,case_10_hotspot_top50,0.372255,0.367360,0.007401,199.761170
2,case_09_min_count_10,0.366054,0.360910,0.009188,166.474774
3,case_05_both_burdens,0.362392,0.360748,0.007648,167.604331
4,case_03_gene_burden,0.362379,0.359619,0.006767,114.505746
5,case_08_min_count_5,0.361211,0.359942,0.007664,163.490306
6,case_07_min_count_3,0.361180,0.359942,0.007595,162.876350
7,case_04_token_burden,0.360935,0.358652,0.006160,137.975805
8,case_01_wt_binary,0.344525,0.342848,0.008874,63.831415
9,case_02_remove_constant,0.344392,0.342687,0.008806,62.441958


## 유망 후보 반복 검증

`best_cases`를 위 결과에서 고른 후보 이름으로 수정합니다. `confirmation=True`는 seed 42/52/62에서 각각 5-Fold를 실행합니다.

In [6]:
best_cases = ["case_05_both_burdens"]

confirmed_leaderboard = run(
    selected_cases=best_cases,
    confirmation=True,
)
confirmed_leaderboard[
    [
        "preprocessing",
        "oof_f1_macro_mean",
        "oof_f1_macro_std",
        "oof_accuracy_mean",
        "oof_accuracy_std",
    ]
]


===== case_05_both_burdens =====
[logistic] seed=42 fold=1/5 f1_macro=0.37320 features=4,232 time=34.7s
[logistic] seed=42 fold=2/5 f1_macro=0.35238 features=4,231 time=30.2s
[logistic] seed=42 fold=3/5 f1_macro=0.35978 features=4,232 time=34.1s
[logistic] seed=42 fold=4/5 f1_macro=0.35806 features=4,230 time=35.0s
[logistic] seed=42 fold=5/5 f1_macro=0.36179 features=4,228 time=31.8s
[logistic] seed=42 OOF Macro F1=0.36239, Accuracy=0.36075
[logistic] seed=52 fold=1/5 f1_macro=0.35942 features=4,231 time=34.6s
[logistic] seed=52 fold=2/5 f1_macro=0.36965 features=4,231 time=35.5s
[logistic] seed=52 fold=3/5 f1_macro=0.34783 features=4,230 time=36.0s
[logistic] seed=52 fold=4/5 f1_macro=0.33252 features=4,231 time=32.3s
[logistic] seed=52 fold=5/5 f1_macro=0.34860 features=4,231 time=36.2s
[logistic] seed=52 OOF Macro F1=0.35463, Accuracy=0.35462
[logistic] seed=62 fold=1/5 f1_macro=0.33530 features=4,232 time=31.7s
[logistic] seed=62 fold=2/5 f1_macro=0.36760 features=4,228 time=32.5

,preprocessing,oof_f1_macro_mean,oof_f1_macro_std,oof_accuracy_mean,oof_accuracy_std
0,case_05_both_burdens,0.358138,0.003935,0.357147,0.003202
